In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.dbutils import *
from datetime import datetime, date

In [0]:
## batting_raw

df_bat_raw = spark.table('ipl_database.bronze.batsman_data_raw').select("batsman","batting_team","date").distinct()
df_ball_raw = spark.table('ipl_database.bronze.bowler_raw_data').select("bowler","bowling_team","date").distinct()

In [0]:
%skip
# df_bat_raw.display()

## 00 Batsman clean batsman name column and trim for leading and trailing blank spaces

In [0]:
df_bat_raw = df_bat_raw.withColumn(
    "player_name",
    regexp_replace(col("batsman"),r'\(c\)',"")
) \
.withColumn(
    "player_name",
    regexp_replace(col("player_name"),"�","")
) \
.withColumn(
    "captain",
    when(col("batsman").contains("(c)"),col("player_name")).otherwise("-")
) \
.withColumn(
    "team",
    trim(col("batting_team"))
) \
.withColumn(
    "role",
    lit("Batsman")
) \
.withColumn(
    "season",
    year(to_date(trim("date"),"MMMM dd yyyy"))
).select("team","player_name","role","season")

In [0]:
%skip
df_bat_raw.display()

## 01 Bowler raw data

In [0]:
df_ball_raw = df_ball_raw.withColumn(
    "team",
    trim(col("bowling_team"))
) \
.withColumn(
    "player_name",
    trim(col("bowler"))
) \
.withColumn(
    "season",
    year(to_date(trim("date"),"MMMM dd yyyy"))
) \
.withColumn(
    "role",
    lit("Bowler")
).select("team","player_name","role","season")

In [0]:
%skip
df_ball_raw.display()

## 02 Append bowler and batsman data

In [0]:
df_ball_raw.createOrReplaceTempView("df_ball_raw")
df_bat_raw.createOrReplaceTempView("df_bat_raw")

In [0]:
remove_bowlers = df_bat_raw.alias('a') \
    .join(
        df_ball_raw.alias('b'),
        (col('a.player_name') == col('b.player_name')) & (col('a.season') == col('b.season')),
        'leftanti'
    ) \
    .select('a.*')

In [0]:
%skip
remove_bowlers.display()

In [0]:
df_fin = remove_bowlers.unionAll(df_ball_raw).orderBy("team").distinct()
# df_fin.display()

In [0]:
try:
    if spark.catalog.tableExists("ipl_database.silver.player_details"):
        df_fin.createOrReplaceTempView('df_fin')
        spark.sql('''
                merge into ipl_database.silver.player_details pd
                using df_fin df
                on pd.team = df.team and pd.player_name = df.player_name and pd.season = df.season
                -- when matched then update set *
                when not matched then insert *
                ''')
        print("Data Inserted Sucessfully!")
    else:
        df_fin.write.format("delta").mode("overwrite").save(f"abfss://silver@ipldatastorageaccount.dfs.core.windows.net/player_details/")
        spark.sql('''
                create table ipl_database.silver.player_details
                using delta
                location "abfss://silver@ipldatastorageaccount.dfs.core.windows.net/player_details/"
                ''')
        print("Data Written Sucessfully!")

except Exception as e:
    print(f"Error while write/append {e}")

## 02 finding all rounder
 - batsman who never bowled will be pure batters else they will be marked all rounders

In [0]:
%skip
df_allrounder = df_bat_raw.alias('a') \
    .join(
        df_ball_raw.alias('b'),
        (col('a.player_name') == col('b.player_name')) & (col('a.season') == col('b.season')),
        'left'
    ) \
    .withColumn(
        'is_allrounder',
        when(col('a.player_name') == col('b.player_name'),lit(1)).otherwise(lit(0))
    ) \
    .select("a.team","a.player_name","a.role","a.season","is_allrounder")
df_allrounder.display()